# MedSpan — Biomedical Extractive QA with BioBERT

**Model:** `dmis-lab/biobert-base-cased-v1.2` fine-tuned for extractive QA
via HuggingFace `AutoModelForQuestionAnswering`.

**Dataset:** [`kroshan/BioASQ`](https://huggingface.co/datasets/kroshan/BioASQ) — each row is a biomedical `question` plus a `text` field encoded as
`<answer> ANSWER <context> CONTEXT`. We split that into `answer` / `context` and
then treat it as standard SQuAD-style extractive QA.

**Runtime targets:**
- Google Colab T4 — full dataset, CUDA
- Apple M1 Mac — 100-example debug subset, MPS

## 0. Install dependencies

In [1]:
import sys, subprocess, importlib.metadata as m

# ---------------------------------------------------------------------------
# Colab T4 currently ships an INCOMPATIBLE pair: torch 2.2.2 + transformers 4.46.3.
# transformers 4.45+ calls `torch.library.register_fake`, which only exists in
# torch 2.4+. We do NOT touch torch (re-installing it breaks Colab's CUDA build),
# so we instead pin transformers (and friends) to the last release line that
# works against torch 2.2.x.
# ---------------------------------------------------------------------------
REQUIRED = {
    "transformers": "4.44.2",   # last 4.44 release; pre-dates the register_fake call
    "accelerate":   "0.34.2",   # ships clear_device_cache; pairs with transformers 4.44
    "datasets":     "2.21.0",   # pairs with transformers 4.44
    "tokenizers":   "0.19.1",   # required by transformers 4.44
    "evaluate":     "0.4.3",
    "gradio":       "4.44.0",
}

missing, wrong = [], []
for pkg, want in REQUIRED.items():
    try:
        got = m.version(pkg)
        if got != want:
            wrong.append((pkg, got, want))
    except m.PackageNotFoundError:
        missing.append(f"{pkg}=={want}")

if not missing and not wrong:
    print("All required packages already at correct versions — skipping pip install.\n")
else:
    if wrong:
        print("Version mismatches detected:")
        for pkg, got, want in wrong:
            print(f"  {pkg}: have {got}, want {want}")
    if missing:
        print(f"Missing packages: {missing}")

    to_install = missing + [f"{pkg}=={want}" for pkg, _, want in wrong]
    print(f"\nInstalling: {to_install}\n")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + to_install, check=True)

    # Downgrades require killing the kernel — modules cached in memory are stale.
    # Colab auto-reconnects after the SIGKILL; re-run this cell once and it
    # will see versions match and proceed straight to the import check.
    print("Install complete. Restarting kernel to clear cached modules …")
    print("After Colab reconnects (a few seconds), re-run this cell.\n")
    import os, time
    time.sleep(2)
    os.kill(os.getpid(), 9)

# ---------------------------------------------------------------------------
# Verify the critical imports actually work before letting the notebook proceed.
# ---------------------------------------------------------------------------
from transformers import Trainer, AutoModelForQuestionAnswering   # noqa: F401
from accelerate.utils.memory import clear_device_cache             # noqa: F401
import transformers, accelerate, datasets, torch

print(f"  torch        {torch.__version__}  (CUDA: {torch.cuda.is_available()})")
print(f"  transformers {transformers.__version__}")
print(f"  accelerate   {accelerate.__version__}")
print(f"  datasets     {datasets.__version__}")
print("\nReady to continue.")

All required packages already at correct versions — skipping pip install.

  torch        2.10.0+cu128  (CUDA: True)
  transformers 4.44.2
  accelerate   0.34.2
  datasets     2.21.0

Ready to continue.


## 1. Imports & device detection

In [2]:
import os
import json
import collections
import numpy as np
import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# Device selection
# ---------------------------------------------------------------------------
# CUDA  → full dataset on Colab T4
# MPS   → debug subset on Apple M1/M2 (Metal Performance Shaders)
# CPU   → fallback for any other machine

if torch.cuda.is_available():
    DEVICE      = "cuda"
    DEBUG_MODE  = False          # use full dataset
    DEBUG_N     = None
    print(f"CUDA detected: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    DEVICE      = "mps"
    DEBUG_MODE  = True           # keep only 100 examples to stay fast
    DEBUG_N     = 100
    print("Apple MPS detected — running in debug mode (100 examples).")
else:
    DEVICE      = "cpu"
    DEBUG_MODE  = True
    DEBUG_N     = 100
    print("No GPU detected — running in debug mode on CPU.")

print(f"Active device : {DEVICE}")
print(f"Debug mode    : {DEBUG_MODE}")

CUDA detected: Tesla T4
Active device : cuda
Debug mode    : False


## 2. Configuration

In [3]:
# ---------------------------------------------------------------------------
# All tuneable hyper-parameters live here so nothing is buried in the code.
# ---------------------------------------------------------------------------

MODEL_NAME   = "dmis-lab/biobert-base-cased-v1.2"
SAVE_DIR     = "./saved_model"

MAX_LENGTH   = 384   # max total token length (question + context)
DOC_STRIDE   = 128   # overlap between windows when context is longer than MAX_LENGTH
BATCH_SIZE   = 16    # per-device batch size; reduce to 8 if you hit OOM

# Training arguments — sensible defaults for a single-GPU Colab run
LEARNING_RATE      = 2e-5
NUM_EPOCHS         = 3 if not DEBUG_MODE else 1
WARMUP_RATIO       = 0.1
WEIGHT_DECAY       = 0.01

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Model : {MODEL_NAME}")
print(f"Epochs: {NUM_EPOCHS}  |  Batch: {BATCH_SIZE}  |  MaxLen: {MAX_LENGTH}  |  Stride: {DOC_STRIDE}")

Model : dmis-lab/biobert-base-cased-v1.2
Epochs: 3  |  Batch: 16  |  MaxLen: 384  |  Stride: 128


## 3. Load the BioASQ dataset

In [4]:
# ---------------------------------------------------------------------------
# kroshan/BioASQ packs each row into just two fields:
#   question : the biomedical factoid question
#   text     : "<answer> ANSWER <context> CONTEXT"   (literal tags, space-separated,
#              no closing tags)
#
# The next cell splits `text` into separate `answer` / `context` columns and
# assigns a synthetic question_id.
# ---------------------------------------------------------------------------

print("Downloading BioASQ dataset …")
raw_dataset = load_dataset(
    "kroshan/BioASQ",
    trust_remote_code=True,
)
print(raw_dataset)
print("\nSample keys:", list(raw_dataset["train"][0].keys()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/3266 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4950 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'text'],
        num_rows: 3266
    })
    validation: Dataset({
        features: ['question', 'text'],
        num_rows: 4950
    })
})

Sample keys: ['question', 'text']


## 4. Explore & filter to factoid questions

In [5]:
# Inspect one raw example to understand the schema
sample = raw_dataset["train"][0]
print(json.dumps(
    {k: (v[:200] if isinstance(v, str) else v) for k, v in sample.items()},
    indent=2,
))

{
  "question": "What is the inheritance pattern of Li\u2013Fraumeni syndrome?",
  "text": "<answer> autosomal dominant <context> Balanced t(11;15)(q23;q15) in a TP53+/+ breast cancer patient from a Li-Fraumeni syndrome family. Li-Fraumeni Syndrome (LFS) is characterized by early-onset carci"
}


In [6]:
import re

# ---------------------------------------------------------------------------
# kroshan/BioASQ format: every row's `text` field looks like
#     <answer> autosomal dominant <context> Balanced t(11;15)(q23;q15) …
# Literal `<answer>` / `<context>` tags, a single space after each tag, and
# NO closing tags. We split on those two markers, falling back to `text` as
# the context if the markers are missing (very rare / malformed rows).
#
# This cell is idempotent — if re-run after the `text` column has already
# been replaced with `context` + `answer`, it detects that and no-ops.
# ---------------------------------------------------------------------------

# Matches `<answer> … <context> …` — DOTALL lets the context span newlines.
_BIOASQ_PATTERN = re.compile(r"<answer>\s*(.*?)\s*<context>\s*(.*)", re.DOTALL)


def parse_bioasq_text(example, idx):
    """Split `text` into `answer` + `context`; add a synthetic question_id."""
    text = example.get("text", "") or ""

    m = _BIOASQ_PATTERN.match(text)
    if m:
        answer_text  = m.group(1).strip()
        context_text = m.group(2).strip()
    else:
        # Strip the marker tags even when only one (or neither) is present,
        # so downstream tokenisation never sees `<answer>` / `<context>`.
        answer_text  = ""
        context_text = re.sub(r"<(answer|context)>", " ", text).strip()

    return {
        "question_id": f"qid_{idx}",
        "question":    example["question"],
        "context":     context_text,
        "answer":      answer_text,
    }


# Idempotency guard — re-running this cell (without re-running the loader)
# must not crash. If the expected new columns already exist, skip the map.
_needs_parse = "text" in raw_dataset["train"].column_names

if _needs_parse:
    print("Parsing 'text' into 'answer' + 'context' + 'question_id' …")
    factoid_dataset = raw_dataset.map(
        parse_bioasq_text,
        with_indices=True,
        remove_columns=raw_dataset["train"].column_names,
        desc="Parsing raw text field",
    )
else:
    print("`text` column already parsed — reusing existing factoid_dataset.")
    factoid_dataset = raw_dataset  # assumed to already carry the parsed columns

# Sanity check: how many rows parsed cleanly?
for split, ds in factoid_dataset.items():
    n_total    = len(ds)
    n_with_ans = sum(1 for a in ds["answer"] if a)
    print(f"{split:>10}: {n_total} rows  |  {n_with_ans} with non-empty answer")

print("\nSample parsed row:")
print({k: (v[:160] + '…') if isinstance(v, str) and len(v) > 160 else v
       for k, v in factoid_dataset['train'][0].items()})

Parsing 'text' into 'answer' + 'context' + 'question_id' …


Parsing raw text field:   0%|          | 0/3266 [00:00<?, ? examples/s]

Parsing raw text field:   0%|          | 0/4950 [00:00<?, ? examples/s]

     train: 3266 rows  |  3264 with non-empty answer
validation: 4950 rows  |  4948 with non-empty answer

Sample parsed row:
{'question': 'What is the inheritance pattern of Li–Fraumeni syndrome?', 'question_id': 'qid_0', 'context': 'Balanced t(11;15)(q23;q15) in a TP53+/+ breast cancer patient from a Li-Fraumeni syndrome family. Li-Fraumeni Syndrome (LFS) is characterized by early-onset car…', 'answer': 'autosomal dominant'}


In [7]:
# Optional: truncate to DEBUG_N examples per split when in debug mode
if DEBUG_MODE and DEBUG_N is not None:
    factoid_dataset = DatasetDict({
        split: ds.select(range(min(DEBUG_N, len(ds))))
        for split, ds in factoid_dataset.items()
    })
    print("Debug subset sizes:", {k: len(v) for k, v in factoid_dataset.items()})

## 5. Preprocess — locate answer spans in context

In [8]:
# ---------------------------------------------------------------------------
# kroshan/BioASQ gives us the answer string but NOT its character offset in
# the context. Extractive QA needs (start, end) positions, so we locate the
# first case-insensitive occurrence of the answer inside the context.
#
# For many rows the answer won't appear verbatim in the provided context —
# those rows aren't trainable as an extractive QA problem and are dropped.
# ---------------------------------------------------------------------------

def add_answer_positions(example):
    context     = example["context"]
    answer_text = example["answer"]

    if not answer_text or not context:
        example["answer_text"]  = answer_text
        example["answer_start"] = -1
        return example

    start_char = context.lower().find(answer_text.lower())
    if start_char != -1:
        # Preserve the original-case slice so training labels stay faithful.
        actual = context[start_char: start_char + len(answer_text)]
    else:
        actual = answer_text

    example["answer_text"]  = actual
    example["answer_start"] = start_char
    return example


factoid_dataset = factoid_dataset.map(
    add_answer_positions,
    desc="Locating answer spans",
)

before = {k: len(v) for k, v in factoid_dataset.items()}
factoid_dataset = factoid_dataset.filter(
    lambda ex: ex["answer_start"] != -1,
    desc="Dropping unanswerable examples",
)
after = {k: len(v) for k, v in factoid_dataset.items()}

for split in before:
    kept    = after.get(split, 0)
    dropped = before[split] - kept
    pct     = (100.0 * dropped / before[split]) if before[split] else 0.0
    print(f"{split:>10}: {kept} kept, {dropped} dropped ({pct:.1f}% unanswerable)")

Locating answer spans:   0%|          | 0/3266 [00:00<?, ? examples/s]

Locating answer spans:   0%|          | 0/4950 [00:00<?, ? examples/s]

Dropping unanswerable examples:   0%|          | 0/3266 [00:00<?, ? examples/s]

Dropping unanswerable examples:   0%|          | 0/4950 [00:00<?, ? examples/s]

     train: 3264 kept, 2 dropped (0.1% unanswerable)
validation: 4948 kept, 2 dropped (0.0% unanswerable)


In [9]:
# Inspect a preprocessed example
ex = factoid_dataset["train"][0]
print("Question    :", ex["question"])
print("Answer text :", ex["answer_text"])
print("Answer start:", ex["answer_start"])
print("Context [:300]:", ex["context"][:300])

Question    : What is the inheritance pattern of Li–Fraumeni syndrome?
Answer text : autosomal dominant
Answer start: 213
Context [:300]: Balanced t(11;15)(q23;q15) in a TP53+/+ breast cancer patient from a Li-Fraumeni syndrome family. Li-Fraumeni Syndrome (LFS) is characterized by early-onset carcinogenesis involving multiple tumor types and shows autosomal dominant inheritance. Approximately 70% of LFS cases are due to germline muta


## 6. Tokenize with stride handling

In [10]:
print("Loading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded:", tokenizer.__class__.__name__)

Loading tokenizer …


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizer loaded: BertTokenizerFast


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [11]:
# ---------------------------------------------------------------------------
# Tokenization for extractive QA
# ---------------------------------------------------------------------------
# When the context is longer than MAX_LENGTH - (question tokens) - 3 special
# tokens, HuggingFace splits it into overlapping windows ("features").
#
# return_overflowing_tokens=True  → multiple features per example
# return_offsets_mapping=True     → char-to-token offset mapping
# stride=DOC_STRIDE               → overlap between consecutive windows
#
# We compute (start_positions, end_positions) labels for both training and
# validation features. Validation additionally keeps offset_mapping +
# example_id so post-processing can map logits → answer spans across all
# windows of an example.
# ---------------------------------------------------------------------------

def tokenize_and_align(examples, is_train=True):
    """Tokenise a batch and compute start/end token labels per feature."""
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",        # only truncate context, never question
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping  = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping  = tokenized["offset_mapping"]   # keep for both train & val

    start_positions = []
    end_positions   = []
    example_ids     = []
    val_offsets     = []     # cleaned offset map for validation post-processing

    for feature_idx, sample_idx in enumerate(sample_mapping):
        # CLS token = "no answer" label position
        input_ids = tokenized["input_ids"][feature_idx]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        # 0 = question tokens, 1 = context tokens, None = specials/padding
        seq_ids = tokenized.sequence_ids(feature_idx)
        offsets = offset_mapping[feature_idx]

        example_ids.append(examples["question_id"][sample_idx])

        ans_start = examples["answer_start"][sample_idx]
        ans_text  = examples["answer_text"][sample_idx]
        ans_end   = ans_start + len(ans_text) - 1   # inclusive char index

        # Find the context window's token boundaries
        ctx_start_tok = next((i for i, s in enumerate(seq_ids) if s == 1), None)
        if ctx_start_tok is None:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            if not is_train:
                val_offsets.append([(0, 0) for _ in offsets])
            continue

        ctx_end_tok = (
            len(seq_ids)
            - next(i for i, s in enumerate(reversed(seq_ids)) if s == 1)
            - 1
        )

        window_start_char = offsets[ctx_start_tok][0]
        window_end_char   = offsets[ctx_end_tok][1]

        # Answer falls outside this window → mark unanswerable
        if ans_start < window_start_char or ans_end > window_end_char:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            # Walk forward to the token that starts the answer
            tok_start = ctx_start_tok
            while tok_start <= ctx_end_tok and offsets[tok_start][0] <= ans_start:
                tok_start += 1
            start_positions.append(tok_start - 1)

            # Walk backward to the token that ends the answer
            tok_end = ctx_end_tok
            while tok_end >= ctx_start_tok and offsets[tok_end][1] >= ans_end:
                tok_end -= 1
            end_positions.append(tok_end + 1)

        if not is_train:
            # For validation: zero out offsets of non-context tokens so the
            # post-processor can ignore question / special / padding positions.
            val_offsets.append([
                o if seq_ids[k] == 1 else (0, 0)
                for k, o in enumerate(offsets)
            ])

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"]   = end_positions
    if is_train:
        # Drop offset_mapping for training — Trainer would complain about extra cols
        tokenized.pop("offset_mapping", None)
    else:
        tokenized["offset_mapping"] = val_offsets

    tokenized["example_id"] = example_ids
    return tokenized


# ---------------------------------------------------------------------------
# Build train and validation feature datasets
# ---------------------------------------------------------------------------
print("Tokenizing training split …")
train_features = factoid_dataset["train"].map(
    lambda ex: tokenize_and_align(ex, is_train=True),
    batched=True,
    remove_columns=factoid_dataset["train"].column_names,
    desc="Tokenising train",
)

# kroshan/BioASQ exposes 'train' and 'validation' — fall back to 'test' if absent.
val_split = "validation" if "validation" in factoid_dataset else "test"
print(f"Tokenizing {val_split} split …")
val_features = factoid_dataset[val_split].map(
    lambda ex: tokenize_and_align(ex, is_train=False),
    batched=True,
    remove_columns=factoid_dataset[val_split].column_names,
    desc="Tokenising validation",
)

print(f"Train features : {len(train_features)}")
print(f"Val features   : {len(val_features)}")
print("Train cols:", train_features.column_names)
print("Val   cols:", val_features.column_names)

Tokenizing training split …


Tokenising train:   0%|          | 0/3264 [00:00<?, ? examples/s]

Tokenizing validation split …


Tokenising validation:   0%|          | 0/4948 [00:00<?, ? examples/s]

Train features : 5500
Val features   : 8256
Train cols: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions', 'example_id']
Val   cols: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'start_positions', 'end_positions', 'example_id']


In [12]:
# Trainer expects torch tensors, not python lists; set the format accordingly.
# Keep example_id as a plain python column (not a tensor) for post-processing.
TRAINER_COLS = ["input_ids", "attention_mask", "token_type_ids",
                "start_positions", "end_positions"]

# token_type_ids may not be present for some tokenisers
trainer_train_cols = [c for c in TRAINER_COLS if c in train_features.column_names]
train_features.set_format(type="torch", columns=trainer_train_cols)

VAL_TENSOR_COLS = ["input_ids", "attention_mask", "token_type_ids",
                   "start_positions", "end_positions"]
trainer_val_cols = [c for c in VAL_TENSOR_COLS if c in val_features.column_names]
val_features.set_format(type="torch", columns=trainer_val_cols, output_all_columns=True)

print("Format set.")

Format set.


## 7. Load BioBERT for Question Answering

In [13]:
print("Loading BioBERT …")
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
model.to(DEVICE)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading BioBERT …


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.2 and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model parameters: 107,721,218


## 8. Evaluation metrics — Exact Match & token-level F1

In [14]:
# ---------------------------------------------------------------------------
# Standard SQuAD-style evaluation helpers.
# These operate on plain text strings, not token ids.
# ---------------------------------------------------------------------------

import string, re

def normalize_answer(s: str) -> str:
    """Lower-case, strip punctuation and articles, collapse whitespace."""
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = " ".join(s.split())
    return s


def exact_match_score(prediction: str, ground_truth: str) -> float:
    return float(normalize_answer(prediction) == normalize_answer(ground_truth))


def f1_score(prediction: str, ground_truth: str) -> float:
    pred_tokens  = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()
    common = collections.Counter(pred_tokens) & collections.Counter(truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall    = num_same / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)


print("Metric helpers defined.")

Metric helpers defined.


In [15]:
# ---------------------------------------------------------------------------
# Post-processing: convert logit outputs → answer strings
# ---------------------------------------------------------------------------
# The Trainer gives us (start_logits, end_logits) for every feature (window).
# For each original example we pick the best non-null span across all windows.
# ---------------------------------------------------------------------------

N_BEST   = 20     # candidate (start, end) pairs to consider per feature
MAX_SPAN = 30     # maximum answer length in tokens

def postprocess_qa_predictions(
    examples,           # original (un-tokenised) examples (HF Dataset of factoid items)
    features,           # tokenised features Dataset (with offset_mapping + example_id)
    raw_predictions,    # (start_logits, end_logits) numpy arrays [num_features, seq_len]
):
    """
    For each example, select the best answer span across all its windows.
    Returns dict {example_id: predicted_answer_string}.
    """
    start_logits, end_logits = raw_predictions

    # Pull the helper columns out once (avoids per-row Dataset lookups in the loop)
    feature_example_ids = features["example_id"]
    feature_offsets     = features["offset_mapping"]

    # Map example_id → list of feature indices
    example_to_features = collections.defaultdict(list)
    for i, ex_id in enumerate(feature_example_ids):
        example_to_features[ex_id].append(i)

    predictions = {}

    for ex in examples:
        ex_id   = ex["question_id"]
        context = ex["context"]
        best_answer = {"text": "", "score": float("-inf")}

        for feat_idx in example_to_features[ex_id]:
            start_log = list(start_logits[feat_idx])
            end_log   = list(end_logits[feat_idx])
            offsets   = feature_offsets[feat_idx]

            # Build a context mask from the offsets: the question side has been
            # zeroed out by the tokenizer (offsets == (0, 0) at specials/padding).
            # We treat any token after the first non-zero offset run as context.
            ctx_mask = [1 if (o[0] != 0 or o[1] != 0) else 0 for o in offsets]

            # Mask out non-context positions in the logits
            for i in range(len(start_log)):
                if not ctx_mask[i]:
                    start_log[i] = float("-inf")
                    end_log[i]   = float("-inf")

            # Top-N start and end positions
            top_starts = sorted(range(len(start_log)), key=lambda i: start_log[i], reverse=True)[:N_BEST]
            top_ends   = sorted(range(len(end_log)),   key=lambda i: end_log[i],   reverse=True)[:N_BEST]

            for start in top_starts:
                for end in top_ends:
                    if (
                        end < start
                        or (end - start + 1) > MAX_SPAN
                        or offsets[start] == (0, 0)
                        or offsets[end]   == (0, 0)
                    ):
                        continue
                    score = start_log[start] + end_log[end]
                    if score > best_answer["score"]:
                        char_start = offsets[start][0]
                        char_end   = offsets[end][1]
                        best_answer = {
                            "text":  context[char_start:char_end],
                            "score": score,
                        }

        predictions[ex_id] = best_answer["text"] if best_answer["text"] else ""

    return predictions


print("Post-processing function defined.")

Post-processing function defined.


In [16]:
# ---------------------------------------------------------------------------
# compute_metrics is called by Trainer after each evaluation epoch.
# eval_pred contains (predictions, label_ids); for QA, predictions is a tuple
# (start_logits, end_logits) and label_ids is (start_positions, end_positions).
# ---------------------------------------------------------------------------

# We capture val_features and factoid_dataset via closure.
val_examples = factoid_dataset[val_split]

def compute_metrics(eval_pred):
    start_logits, end_logits = eval_pred.predictions

    preds = postprocess_qa_predictions(
        examples=val_examples,
        features=val_features,
        raw_predictions=(start_logits, end_logits),
    )

    em_scores, f1_scores = [], []
    for ex in val_examples:
        ex_id   = ex["question_id"]
        gold    = ex["answer_text"]
        pred    = preds.get(ex_id, "")
        em_scores.append(exact_match_score(pred, gold))
        f1_scores.append(f1_score(pred, gold))

    return {
        "exact_match": round(100.0 * np.mean(em_scores), 2),
        "f1":          round(100.0 * np.mean(f1_scores), 2),
    }


print("compute_metrics defined.")

compute_metrics defined.


## 9. Fine-tuning with HuggingFace Trainer

In [17]:
training_args = TrainingArguments(
    output_dir="./checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=2 if torch.cuda.is_available() else 0,
    save_total_limit=2,
)

print("TrainingArguments updated with metric_for_best_model='f1'.")

TrainingArguments updated with metric_for_best_model='f1'.


In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_features,
    eval_dataset=val_features,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer initialised. Starting training …")
train_result = trainer.train()

print("\nTraining complete.")
print(f"  Train loss     : {train_result.training_loss:.4f}")
print(f"  Train runtime  : {train_result.metrics['train_runtime']:.1f}s")

Trainer initialised. Starting training …


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Exact Match,F1
1,0.579000,0.827871,44.020000,46.970000
2,0.234300,0.862449,44.540000,47.810000
3,0.142400,0.869322,45.370000,48.520000



Training complete.
  Train loss     : 0.7526
  Train runtime  : 529.4s


## 10. Evaluate on validation set

In [19]:
eval_metrics = trainer.evaluate()

print("\n===== Evaluation Results =====")
for k, v in eval_metrics.items():
    print(f"  {k}: {v}")


===== Evaluation Results =====
  eval_loss: 0.8693219423294067
  eval_exact_match: 45.37
  eval_f1: 48.52
  eval_runtime: 59.9592
  eval_samples_per_second: 137.694
  eval_steps_per_second: 8.606
  epoch: 3.0


## 11. Save the fine-tuned model

In [20]:
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model and tokenizer saved to {SAVE_DIR}")

Model and tokenizer saved to ./saved_model


## 12. Inference function

In [21]:
from transformers import pipeline as hf_pipeline

qa_pipe = hf_pipeline(
    "question-answering",
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0 if DEVICE == "cuda" else -1,   # -1 → CPU/MPS handled automatically
    handle_impossible_answer=False,
)


def predict_answer(question: str, context: str) -> dict:
    """
    Given a biomedical question and a PubMed abstract, return the
    extracted answer span and confidence score.

    Args:
        question : biomedical factoid question
        context  : PubMed abstract text

    Returns:
        dict with keys 'answer' (str) and 'score' (float)
    """
    result = qa_pipe(question=question, context=context)
    return {"answer": result["answer"], "score": round(result["score"], 4)}


print("Inference function ready.")

Inference function ready.


## 13. Demo predictions

In [22]:
# Run inference on the first 3 validation examples
print("Sample predictions on validation examples:\n")
for i in range(min(3, len(val_examples))):
    ex = val_examples[i]
    result = predict_answer(ex["question"], ex["context"])
    print(f"[{i+1}] Question  : {ex['question']}")
    print(f"    Gold answer: {ex['answer_text']}")
    print(f"    Prediction : {result['answer']}  (score: {result['score']})")
    print()

Sample predictions on validation examples:

[1] Question  : Name synonym of Acrokeratosis paraneoplastica.
    Gold answer: Bazex syndrome
    Prediction : Acrokeratosis paraneoplastica (Bazex syndrome  (score: 0.1581)

[2] Question  : Name synonym of Acrokeratosis paraneoplastica.
    Gold answer: Bazex syndrome
    Prediction : Acrokeratosis paraneoplastica (Bazex syndrome  (score: 0.1581)

[3] Question  : Name synonym of Acrokeratosis paraneoplastica.
    Gold answer: Bazex syndrome
    Prediction : Bazex syndrome  (score: 0.4854)



In [23]:
# ---------------------------------------------------------------------------
# Custom inference — paste your own question and abstract here
# ---------------------------------------------------------------------------
MY_QUESTION = "What is the primary mechanism of action of metformin?"

MY_ABSTRACT = (
    "Metformin is a biguanide drug used as a first-line treatment for type 2 "
    "diabetes mellitus. Its primary mechanism of action involves the inhibition "
    "of hepatic glucose production (gluconeogenesis) via activation of AMP-activated "
    "protein kinase (AMPK). Additionally, metformin improves peripheral insulin "
    "sensitivity and reduces intestinal glucose absorption."
)

output = predict_answer(MY_QUESTION, MY_ABSTRACT)
print(f"Question : {MY_QUESTION}")
print(f"Answer   : {output['answer']}")
print(f"Score    : {output['score']}")

Question : What is the primary mechanism of action of metformin?
Answer   : Metformin is a biguanide
Score    : 0.002
